# Week 3 – Builder (Cemre Gönenç): OCR cleanup

**Goal:** OCR reads at least **70%** of plates correctly.

Week 2 baseline: PaddleOCR read **40/50 (80%)** with no cleanup. This week we add:
1. **Image cleanup:** grayscale + enlarge 2x (`ocr/plate_reader.py`)
2. **Text cleanup:** remove symbols and the `TR` band, fix O→0, I→1, S→5 … only in number positions (and the reverse in letter positions) (`ocr/plate_text.py`)
3. **Pattern check:** Turkish format = 2 digits + 1–3 letters + 2–5 digits

Then we measure each step on the 50 plates, and on **real crops from Doğukan's plate detector**.

Runtime → Change runtime type → **T4 GPU**. Run cells one by one.

## 1) Install + get the code

In [ ]:
!pip install -q paddlepaddle paddleocr

In [ ]:
import os
if not os.path.exists('anpr-project-en'):
    !git clone -q https://github.com/cemregonenc/anpr-project-en.git
else:
    !cd anpr-project-en && git pull -q
os.chdir('/content/anpr-project-en')
print(os.getcwd())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RESULTS = '/content/drive/MyDrive/ANPR/results'
os.makedirs(RESULTS, exist_ok=True)

import sys, pandas as pd
sys.path.insert(0, 'ocr')
from plate_reader import PlateReader
from plate_text import basic_clean, fix_plate, is_valid_plate

labels = pd.read_csv('data/plates/labels.csv')
reader = PlateReader()      # loads PaddleOCR once (first time downloads the models)
print(len(labels), 'plates')

## 2) Unit tests for the text rules
All must say `ok`. They check that the 50 correct plates match the pattern and that the fixes work.

In [ ]:
!python ocr/test_plate_text.py

## 3) Measure each cleanup step on the 50 plates
| Setting | Image cleanup | Text fixes |
|---|---|---|
| A – Week 2 baseline | no | no (only remove spaces/symbols) |
| B – image only | gray + 2x | no |
| C – text only | no | yes |
| D – **full** | gray + 2x | yes |

In [ ]:
import time
settings = {
    'A_baseline':   dict(gray=False, scale=1, fix=False),
    'B_image_only': dict(gray=True,  scale=2, fix=False),
    'C_text_only':  dict(gray=False, scale=1, fix=True),
    'D_full':       dict(gray=True,  scale=2, fix=True),
}
rows = []
for name, s in settings.items():
    reader.gray, reader.scale, reader.fix = s['gray'], s['scale'], s['fix']
    t0 = time.time()
    out = [reader.read(f'data/plates/{f}') for f in labels['filename']]
    labels[f'{name}_text'] = [o['text'] for o in out]
    labels[f'{name}_ok'] = labels[f'{name}_text'] == labels['plate_text']
    rows.append({'Setting': name,
                 'Correct (of 50)': int(labels[f'{name}_ok'].sum()),
                 'Accuracy %': round(100 * labels[f'{name}_ok'].mean(), 1),
                 'Valid pattern %': round(100 * pd.Series([is_valid_plate(o['text']) for o in out]).mean(), 1),
                 'Time (s)': round(time.time() - t0, 1)})
results = pd.DataFrame(rows)
results

### By condition (clear, blurry, small …)

In [ ]:
cols = [c for c in labels.columns if c.endswith('_ok')]
labels.groupby('condition')[cols].sum().assign(plates=labels.groupby('condition').size()).sort_values('plates', ascending=False)

### Plates the full setting (D) still gets wrong

In [ ]:
import matplotlib.pyplot as plt, cv2
wrong = labels[~labels['D_full_ok']]
print(len(wrong), 'wrong')
if len(wrong):
    cols_n = 3; rows_n = (len(wrong) + 2) // 3
    plt.figure(figsize=(15, 2.6 * rows_n))
    for i, (_, r) in enumerate(wrong.iterrows()):
        img = cv2.cvtColor(cv2.imread(f"data/plates/{r['filename']}"), cv2.COLOR_BGR2RGB)
        plt.subplot(rows_n, cols_n, i + 1); plt.imshow(img); plt.axis('off')
        plt.title(f"truth {r['plate_text']} | read {r['D_full_text'] or '-'}", fontsize=9)
    plt.tight_layout(); plt.show()

## 4) Real crops from Doğukan's script (`models/crop_plates.py`)
Doğukan runs `models/week3_plate_crops.ipynb`, which saves the crops to Drive: `ANPR/auto_crops/` with a `crops.csv`.

- **First run:** this cell creates `ANPR/auto_crops/labels.csv` with an empty `plate_text` column. Open it in Google Sheets / Excel, type the correct plate for each crop (leave empty if unreadable), save as CSV.
- **Second run:** the cell measures accuracy on those real crops.

In [ ]:
AUTO = '/content/drive/MyDrive/ANPR/auto_crops'
lab_path = f'{AUTO}/labels.csv'
if not os.path.exists(f'{AUTO}/crops.csv'):
    print('No auto crops yet – ask Doğukan to run models/week3_plate_crops.ipynb')
else:
    crops = pd.read_csv(f'{AUTO}/crops.csv')
    if not os.path.exists(lab_path):
        crops.assign(plate_text='')[['crop_file', 'plate_text']].to_csv(lab_path, index=False)
        print(f'Created {lab_path} – fill in plate_text, then run this cell again.')
    else:
        reader.gray, reader.scale, reader.fix = True, 2, True
        real = pd.read_csv(lab_path).fillna('')
        real = real[real['plate_text'].str.strip() != '']
        real['plate_text'] = real['plate_text'].apply(basic_clean)
        real['read'] = [reader.read(f'{AUTO}/{f}')['text'] for f in real['crop_file']]
        real['ok'] = real['read'] == real['plate_text']
        print(f"Real crops: {real['ok'].sum()}/{len(real)} correct = {100 * real['ok'].mean():.1f}%")
        real.to_csv(f'{RESULTS}/week3_real_crops.csv', index=False)
        display(real[~real['ok']])

## 5) Save + summary for docs/experiments.md

In [ ]:
labels.to_csv(f'{RESULTS}/week3_ocr_cleanup.csv', index=False)
results.to_csv(f'{RESULTS}/week3_ocr_summary.csv', index=False)
best = results.sort_values('Accuracy %', ascending=False).iloc[0]
print('| ' + ' | '.join(results.columns) + ' |')
print('|' + '---|' * len(results.columns))
for _, r in results.iterrows():
    print('| ' + ' | '.join(str(v) for v in r) + ' |')
print(f"\nBest: {best['Setting']} – {best['Correct (of 50)']}/50 ({best['Accuracy %']}%).  Target 70%: {'PASSED' if best['Accuracy %'] >= 70 else 'NOT YET'}")

## 6) Conclusion (fill in)
- Best setting: …  (use it in the app: `PlateReader(gray=…, scale=…, fix=True)`)
- Did image cleanup help? …
- Did text fixes help? …
- Accuracy on real auto crops: … / … = … %
- Still failing: …